# DQN on CartPole-v1

**Deep Q-Network (DQN)** is an off-policy algorithm that uses a neural network to approximate the action-value function $Q(s, a)$. It employs two key stabilisation tricks:

- **Experience replay**: transitions are stored in a replay buffer and sampled randomly for training.
- **Target network**: a periodically updated copy of the Q-network used to compute stable Bellman targets.

Exploration is driven by **$\varepsilon$-greedy** decay.

> Paper: [Human-level control through deep reinforcement learning](https://www.nature.com/articles/nature14236)

In this notebook we train DQN on **CartPole-v1** (discrete action space) using `lit-sb3`.


## Setup


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

from lit_sb3.algorithms.dqn.dqn import DQN

## Create the model

Key hyperparameters:

| Parameter                | Description                                               |
| ------------------------ | --------------------------------------------------------- |
| `buffer_size`            | Replay buffer capacity                                    |
| `learning_starts`        | Warm-up steps before gradient updates begin               |
| `exploration_fraction`   | Fraction of training over which $\varepsilon$ is annealed |
| `exploration_final_eps`  | Final $\varepsilon$ value                                 |
| `target_update_interval` | Steps between hard target-network updates                 |
| `train_freq`             | Collect env steps between gradient updates                |


In [ ]:
model = DQN(
    "MlpPolicy",
    "CartPole-v1",
    verbose=1,
    learning_rate=1e-4,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=32,
    tau=1.0,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=1_000,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    policy_kwargs={"net_arch": [64, 64]},
)
print(model.policy)

## Train

Each Lightning step collects env transitions and periodically performs gradient updates.


In [ ]:
model.learn(total_timesteps=50_000)

## Evaluate


In [ ]:
eval_env = gym.make("CartPole-v1")
n_eval_episodes = 10
episode_rewards = []

for _ in range(n_eval_episodes):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0.0
    while not done:
        action, _ = model.predict(obs[np.newaxis], deterministic=True)
        obs, reward, terminated, truncated, _ = eval_env.step(int(action[0]))
        total_reward += reward  # ty:ignore[unsupported-operator]        done = terminated or truncated
    episode_rewards.append(total_reward)

print(f"Mean reward over {n_eval_episodes} episodes: {np.mean(episode_rewards):.1f} (± {np.std(episode_rewards):.1f})")